# Train the VANDF → RxNorm bi-encoder (Colab)

This notebook runs the same `train()` as `scripts/06_train.py`, on a Colab GPU. It clones the repo, installs the package, pulls the dataset from the W&B Artifact, trains, evaluates, and logs everything to the `rxnorm-vandf` project.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then add one secret in the key icon (🔑) on the left sidebar, with *Notebook access* on:

- `WANDB_API_KEY`: from https://wandb.ai/authorize

Background reading: `docs/training-primer.md` and `docs/wandb-primer.md` in the repo.

## 1. Check the GPU
Expect a Tesla T4 (or better). If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
A plain clone of the public repo; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone directory goes on `sys.path`, so `import rxnorm_vandf` reads the code straight from the clone (a `git pull` in cell 2 is picked up immediately). An editable `pip install -e` would need a kernel restart to take effect in Colab.

The pip line adds only what Colab doesn't already ship, without upgrading what it does: upgrading Colab's numpy/pandas/torch inside a running kernel breaks its preinstalled stack.

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, numpy, pandas, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets. `wandb.login()` reads the `WANDB_API_KEY` environment variable, so nothing is pasted or printed.

In [ ]:
import wandb
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Configure and train
`TrainConfig` holds every knob (see `docs/training-primer.md` for what each one does). With `data_dir=None` the dataset is downloaded from the W&B Artifact `vandf-rxnorm-pairs:latest`, which also records lineage.

Set `smoke=True` for a 2-minute end-to-end check the first time. The real run is about 10 minutes on a T4. The run URL is printed near the top of the output; open it to watch `train/loss` and the per-epoch `val/acc@1`.

In [ ]:
from rxnorm_vandf.train import TrainConfig, train

cfg = TrainConfig(
    base_model="sentence-transformers/all-MiniLM-L6-v2",
    epochs=4,
    batch_size=64,
    lr=2e-5,
    negatives="ingredient",   # ingredient | tfidf | none
    smoke=False,
    data_dir=None,            # pull the W&B artifact
    output_dir="models",
    run_name="minilm-colab",
    tags=["colab"],
)
best_dir = train(cfg)
print("best model:", best_dir)

## 6. What to look at
- **Runs table** (project page): pin `test/acc@1` and `test/recall@5`; compare this run with `tfidf` (0.509 / 0.820) and with the local-GPU run.
- **Charts:** `train/loss` (should fall then flatten) and `val/acc@1` by epoch (should rise then plateau).
- **Tables → `test/failures`:** did the failure mix change compared with TF-IDF?
- **Artifacts:** `vandf-rxnorm-biencoder` now has a new version whose lineage points at this run and at `vandf-rxnorm-pairs:v0`.

To try a different setting, change `cfg` and rerun cell 5 with a new `run_name`.